In [ ]:
! pip install matplotlib seaborn scikit-learn flask mlflow streamlit

  Using cached matplotlib-3.10.7-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (11 kB)
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached scikit_learn-1.7.2-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (11 kB)
  Using cached flask-3.1.2-py3-none-any.whl.metadata (3.2 kB)
  Using cached mlflow-3.6.0-py3-none-any.whl.metadata (31 kB)
  Using cached contourpy-1.3.3-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.60.1-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (112 kB)
  Using cached kiwisolver-1.4.9-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (6.3 kB)
  Using cached numpy-2.3.4-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached pillow-12.0.0-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata

In [60]:
import json
import mlflow
import requests
import numpy as np
import pandas as pd
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

In [3]:
mlflow.set_experiment("Aula_4")

/home/teru/aula_monitoramento/.venv/lib/python3.11/site-packages/mlflow/tracking/_tracking_service/utils.py:140: FutureWarning: Filesystem tracking backend (e.g., './mlruns') is deprecated. Please switch to a database backend (e.g., 'sqlite:///mlflow.db'). For feedback, see: https://github.com/mlflow/mlflow/issues/18534
  return FileStore(store_uri, store_uri)
2025/11/13 20:57:57 INFO mlflow.tracking.fluent: Experiment with name 'Aula_4' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:///home/teru/aula_monitoramento/mlruns/472781115690188779', creation_time=1763078277018, experiment_id='472781115690188779', last_update_time=1763078277018, lifecycle_stage='active', name='Aula_4', tags={}>

In [ ]:
# Load the Iris dataset
full_dataset = datasets.load_iris(return_X_y=False)
X, y = datasets.load_iris(return_X_y=True)

# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Define the model hyperparameters
params = {
    "solver": "lbfgs",
    "max_iter": 1000,
    "multi_class": "auto",
    "random_state": 8888,
}

In [ ]:
# Start an MLflow run
with mlflow.start_run() as run:
    # Log the hyperparameters
    mlflow.log_params(params)

    # Train the model
    lr = LogisticRegression(**params)
    lr.fit(X_train, y_train)

    # Log the model
    mlflow.sklearn.log_model(sk_model=lr, name="iris_model", input_example=np.array([[5.1, 3.5, 1.4, 0.2]]))

    # Predict on the test set, compute and log the loss metric
    y_pred = lr.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric('f1', f1)

    # Optional: Set a tag that we can use to remind ourselves what this run was for
    mlflow.set_tag("Training Info", "Basic LR model for iris data")
    
    run_id = run.info.run_id

    model_uri = f"runs:/{run.info.run_id}/iris_model"
    mv = mlflow.register_model(
        model_uri, "iris_model", tags={"version": "latest"}
    )
    

/home/teru/aula_monitoramento/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/home/teru/aula_monitoramento/.venv/lib/python3.11/site-packages/mlflow/tracking/_model_registry/utils.py:215: FutureWarning: Filesystem model registry backend (e.g., './mlruns') is deprecated. Please switch to a database backend (e.g., 'sqlite:///mlflow.db'). For feedback, see: https://github.com/mlflow/mlflow/issues/18534
  return FileStore(store_uri)
Successfully registered model 'iris_model'.
2025/11/13 21:02:57 WARNING mlflow.tracking._model_registry.fluent: Run with id c92e7f3720b64382aaa74e968686a2c3 has no artifacts at artifact path 'iris_model', registering model based on models:/m-db759bbeee6a431583d2c9a4d4680a86 instead
Created version '1' of model 'iris_model'.


Rodando o mlflow ui:


``mlflow ui -p 5000``

---

In [71]:
url = "http://localhost:5001/invocations"
url_n = "http://localhost:5001/n_invocations"

In [59]:
payload = {"inputs": [5.1, 3.5, 1.4, 0.2]}  # setosa

pred = requests.post(url, json=payload).json()
print(pred)


{'predictions': 'setosa'}


In [51]:
payload = {"inputs": [6.2, 3.4, 5.4, 2.3]}   # virginica

pred = requests.post(url, json=payload).json()

pred

{'predictions': 'virginica'}

---

In [53]:
csv_path = "datasets/iris_ls.csv"   # ajuste para o nome correto do seu arquivo

# 1. Carregar o CSV
df = pd.read_csv(csv_path)

feature_cols = [
    "sepal length (cm)",
    "sepal width (cm)",
    "petal length (cm)",
    "petal width (cm)"
]

# Detecta automaticamente se a coluna target é numérica ou textual
target_col = "target"

X = df[feature_cols]
y = df[target_col]

# 3. Dividir em treino e teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [55]:
with mlflow.start_run(run_id=run_id, nested=True) as run:
    # Log the hyperparameters
    mlflow.log_params(params)

    # Train the model
    lr = LogisticRegression(**params)
    lr.fit(X_train, y_train)

    # Log the model
    mlflow.sklearn.log_model(sk_model=lr, name="iris_model", input_example=np.array([[5.1, 3.5, 1.4, 0.2]]))

    # Predict on the test set, compute and log the loss metric
    y_pred = lr.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric('f1', f1)

    # Optional: Set a tag that we can use to remind ourselves what this run was for
    mlflow.set_tag("Training Info", "Basic LR model for iris data")
    
    run_id = run.info.run_id

    model_uri = f"runs:/{run.info.run_id}/iris_model"
    mv = mlflow.register_model(
        model_uri, "iris_model", tags={"version": "2"}
    )
    

/home/teru/aula_monitoramento/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/home/teru/aula_monitoramento/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
/home/teru/aula_monitoramento/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
Registered model 'iris_model' already exists. Creating a new version of this model...
2025/11/13 22:30:30 WARNING mlflow.tracking._model_registry.fluent: Run with id c92e7f3720b64382aaa74e968686a2c3 has no artifacts at artifact path 'iris_model', regis

In [56]:
payload = {"inputs": [6.2, 3.4, 5.4, 2.3]}   # virginica

pred = requests.post(url, json=payload).json()

pred

{'predictions': 'virginica'}

In [77]:
payload = {"inputs": [7.0, 3.2, 4.7, 1.4]}   # versicolor

pred = requests.post(url, json=payload).json()

pred

{'predictions': 'versicolor'}

In [82]:
def load_csv(path_csv):
    df_test = pd.read_csv(path_csv)
    feature_cols = [
        "sepal length (cm)",
        "sepal width (cm)",
        "petal length (cm)",
        "petal width (cm)"
    ]
    target_col = "target"
    X = df_test[feature_cols]
    y = df_test[target_col]

    return X, y

In [ ]:
x, y_target = load_csv('datasets/iris_b.csv')

resultados = []

for idx, row in x.iterrows():
    payload = {"inputs": row.values.tolist()}  # transforma a linha em lista

    response = requests.post(url_n, json=payload)

    if response.status_code != 200:
        print(f"Erro na linha {idx}: status {response.status_code}")
        resultados.append(None)
        continue

    try:
        result = response.json()['predictions']
    except json.JSONDecodeError:
        result = None

    resultados.append(int(result))

# Montar DataFrame com resultados
y_real = pd.DataFrame({"predicao": resultados})

In [89]:
f1 = f1_score(y_real, y_target, average='weighted')
f1

0.973344004268374

---

In [90]:
x, y_target = load_csv('datasets/iris_sw.csv')

resultados = []

for idx, row in X.iterrows():
    payload = {"inputs": row.values.tolist()}  # transforma a linha em lista

    response = requests.post(url_n, json=payload)

    if response.status_code != 200:
        print(f"Erro na linha {idx}: status {response.status_code}")
        resultados.append(None)
        continue

    try:
        result = response.json()['predictions']
    except json.JSONDecodeError:
        result = None

    resultados.append(int(result))

# Montar DataFrame com resultados
y_real = pd.DataFrame({"predicao": resultados})

In [91]:
f1 = f1_score(y_real, y_target, average='weighted')
f1

0.33917123463584664

---

In [94]:
log_path="data.log"
csv_path="data.csv"

"""
Lê um arquivo .log linha a linha e salva seu conteúdo em um arquivo .csv.

Cada linha vira um registro no CSV com uma coluna chamada 'content'.
"""
registros = []
# Lê as linhas do arquivo .log
with open(log_path, "r", encoding="utf-8") as f:
    for linha in f:
            # Remove aspas e quebras de linha
            linha = linha.strip().replace('"', "")
            
            # Divide pelos valores
            valores = linha.split(",")
            
            # Converte para float
            valores = [float(v) for v in valores]

            registros.append(valores)

columns = ["sepal length (cm)","sepal width (cm)","petal length (cm)","petal width (cm)","target"]
# Converte para DataFrame
df = pd.DataFrame(registros, columns=columns)

# Salva como CSV
df.to_csv(csv_path, index=False)

print(f"Arquivo salvo como: {csv_path}")

Arquivo salvo como: data.csv
